# Section 5: AI Agents and Intelligent Automation

## Learning Objectives

By the end of this section, you will be able to:
- Distinguish between copilots (suggest) and agents (act)
- Explain the ReAct pattern to stakeholders
- Design appropriate human checkpoints for agent systems
- Recognize common agent failure modes
- Make informed decisions about agent autonomy levels

In [ ]:
#@title Setup (Run this cell first)
%%capture
!pip install "ipywidgets>=7,<8" plotly pandas numpy

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
import time

from google.colab import output
output.enable_custom_widget_manager()

print("Setup complete!")

---
## The $440 Million Mistake

**Knight Capital, 2012:** A software bug in their trading algorithm caused the system to execute erroneous trades autonomously. In **45 minutes**, they lost **$440 million** - nearly bankrupting the company.

The agent had **no human checkpoints** for high-risk actions.

---

**Contrast with Klarna (2024):** Their AI assistant now handles 2/3 of all customer service chats - equivalent to 700 full-time agents. But it took **2 years** of careful trust progression to get there safely.

**The difference?** Klarna started with humans reviewing every response.

---
## Demo 1: Copilot vs Agent Decision Tool

When should AI suggest (copilot) vs act autonomously (agent)?

**Adjust the sliders to see when autonomous agents are appropriate.**

In [ ]:
#@title Copilot vs Agent Analyzer

# Create widgets
action_type = widgets.Dropdown(
    options=[
        ('Draft an email', 'draft_email'),
        ('Send an email', 'send_email'),
        ('Schedule a meeting', 'schedule'),
        ('Make a purchase', 'purchase'),
        ('Delete data', 'delete'),
        ('Transfer money', 'transfer'),
        ('Generate a report', 'report')
    ],
    value='send_email',
    description='Action:',
    style={'description_width': '100px'}
)

undo_difficulty = widgets.IntSlider(
    value=3,
    min=1,
    max=5,
    step=1,
    description='Undo Difficulty:',
    style={'description_width': '100px'},
    continuous_update=False
)

error_cost = widgets.IntSlider(
    value=1000,
    min=0,
    max=100000,
    step=1000,
    description='Error Cost ($):',
    style={'description_width': '100px'},
    continuous_update=False
)

ai_trust = widgets.IntSlider(
    value=3,
    min=1,
    max=5,
    step=1,
    description='AI Trust Level:',
    style={'description_width': '100px'},
    continuous_update=False
)

output_area = widgets.Output()

def analyze_action(action, undo, cost, trust):
    with output_area:
        clear_output(wait=True)

        # Calculate risk score
        risk_score = (undo * 20) + (min(cost, 50000) / 1000) + ((5 - trust) * 10)

        # Determine recommendation
        if risk_score < 30:
            recommendation = "Agent (Autonomous)"
            color = "#22c55e"
            explanation = "Low risk - agent can act independently with logging."
        elif risk_score < 60:
            recommendation = "Agent + Undo Option"
            color = "#40B8A6"
            explanation = "Medium risk - agent can act but provide easy undo."
        elif risk_score < 80:
            recommendation = "Copilot (Suggest Only)"
            color = "#f59e0b"
            explanation = "Higher risk - AI should suggest, human decides."
        else:
            recommendation = "Require Human Approval"
            color = "#dc2626"
            explanation = "High risk - always require explicit approval."

        # Create gauge chart
        fig = go.Figure(go.Indicator(
            mode="gauge+number",
            value=risk_score,
            title={'text': "Risk Score"},
            gauge={
                'axis': {'range': [0, 100]},
                'bar': {'color': color},
                'steps': [
                    {'range': [0, 30], 'color': "#ecfdf5"},
                    {'range': [30, 60], 'color': "#E6F7F5"},
                    {'range': [60, 80], 'color': "#fffbeb"},
                    {'range': [80, 100], 'color': "#fef2f2"}
                ],
                'threshold': {
                    'line': {'color': "black", 'width': 4},
                    'thickness': 0.75,
                    'value': risk_score
                }
            }
        ))

        fig.update_layout(height=300, margin=dict(t=50, b=0))
        fig.show()

        # Display recommendation
        display(HTML(f"""
        <div style="padding: 20px; background: {color}20; border-left: 4px solid {color}; margin-top: 10px;">
            <h3 style="color: {color}; margin: 0;">Recommendation: {recommendation}</h3>
            <p style="margin: 10px 0 0 0;">{explanation}</p>
        </div>
        """))

# Create interactive widget
interactive_output = widgets.interactive_output(
    analyze_action,
    {'action': action_type, 'undo': undo_difficulty, 'cost': error_cost, 'trust': ai_trust}
)

# Display
display(widgets.VBox([
    widgets.HTML("<h4>Configure the Action</h4>"),
    action_type,
    undo_difficulty,
    widgets.HTML("<small>1 = Easy to undo, 5 = Impossible to undo</small>"),
    error_cost,
    ai_trust,
    widgets.HTML("<small>1 = New/untested AI, 5 = Proven track record</small>"),
    output_area
]))

### PM Insight: The Reversibility Test

Before deciding copilot vs agent, ask: **"What's the blast radius if this goes wrong?"**

| Blast Radius | Recommendation |
|--------------|----------------|
| Small (typo in draft) | Agent OK |
| Medium (wrong meeting) | Agent + Undo |
| Large (money sent, data deleted) | Always require approval |

**Discussion:** Think of an AI feature in your product. Where would it fall on this scale?

---
## Demo 2: Trust Progression Simulator

How does an AI system earn more autonomy over time?

**Simulate weeks of operation and see how trust levels change.**

In [ ]:
#@title Trust Progression Over Time

weeks_slider = widgets.IntSlider(
    value=8,
    min=1,
    max=24,
    step=1,
    description='Weeks:',
    style={'description_width': '80px'},
    continuous_update=False
)

error_rate = widgets.FloatSlider(
    value=2.0,
    min=0,
    max=10,
    step=0.5,
    description='Error Rate %:',
    style={'description_width': '80px'},
    continuous_update=False
)

risk_level = widgets.Dropdown(
    options=[('Low Risk Tasks', 'low'), ('Medium Risk Tasks', 'medium'), ('High Risk Tasks', 'high')],
    value='medium',
    description='Task Type:',
    style={'description_width': '80px'}
)

output_area2 = widgets.Output()

def simulate_trust(weeks, errors, risk):
    with output_area2:
        clear_output(wait=True)

        # Risk multipliers
        risk_mult = {'low': 0.5, 'medium': 1.0, 'high': 2.0}[risk]

        # Promotion thresholds
        thresholds = {
            0: {'accuracy': 0, 'weeks': 0},      # Level 0: Start
            1: {'accuracy': 95, 'weeks': 4},     # Level 1: After 4 weeks at 95%
            2: {'accuracy': 98, 'weeks': 8},     # Level 2: After 8 weeks at 98%
            3: {'accuracy': 99, 'weeks': 16}     # Level 3: After 16 weeks at 99%
        }

        # Simulate week by week
        trust_history = []
        current_level = 0
        accuracy = 100 - (errors * risk_mult)

        for week in range(1, weeks + 1):
            # Check for promotion
            if current_level < 3:
                next_level = current_level + 1
                if (week >= thresholds[next_level]['weeks'] and
                    accuracy >= thresholds[next_level]['accuracy']):
                    current_level = next_level

            # Check for demotion (high error rate)
            if errors > 5 and current_level > 0:
                current_level = max(0, current_level - 1)

            trust_history.append({
                'Week': week,
                'Trust Level': current_level,
                'Accuracy': accuracy
            })

        df = pd.DataFrame(trust_history)

        # Create chart
        fig = go.Figure()

        # Add trust level line
        fig.add_trace(go.Scatter(
            x=df['Week'],
            y=df['Trust Level'],
            mode='lines+markers',
            name='Trust Level',
            line=dict(color='#40B8A6', width=3),
            marker=dict(size=8)
        ))

        # Add level zones
        level_names = ['Level 0: Human Does', 'Level 1: Human Approves',
                       'Level 2: Human Reviews', 'Level 3: Autonomous']
        level_colors = ['#fef2f2', '#fffbeb', '#ecfdf5', '#E6F7F5']

        for i in range(4):
            fig.add_hrect(
                y0=i-0.3, y1=i+0.3,
                fillcolor=level_colors[i],
                opacity=0.5,
                line_width=0,
                annotation_text=level_names[i],
                annotation_position="right"
            )

        fig.update_layout(
            title=f"Trust Progression Over {weeks} Weeks (Accuracy: {accuracy:.1f}%)",
            xaxis_title="Week",
            yaxis_title="Trust Level",
            yaxis=dict(range=[-0.5, 3.5], tickmode='array', tickvals=[0, 1, 2, 3]),
            height=400,
            showlegend=False
        )

        fig.show()

        # Current status
        level_desc = level_names[current_level]
        display(HTML(f"""
        <div style="padding: 15px; background: #E6F7F5; border-left: 4px solid #40B8A6; margin-top: 10px;">
            <strong>Current Status:</strong> {level_desc}<br>
            <strong>Accuracy:</strong> {accuracy:.1f}%<br>
            <strong>Next Promotion:</strong> {'Fully autonomous!' if current_level == 3 else f'Need {thresholds[current_level+1]["accuracy"]}% accuracy for {thresholds[current_level+1]["weeks"]} weeks'}
        </div>
        """))

interactive_output2 = widgets.interactive_output(
    simulate_trust,
    {'weeks': weeks_slider, 'errors': error_rate, 'risk': risk_level}
)

display(widgets.VBox([
    widgets.HTML("<h4>Simulation Parameters</h4>"),
    weeks_slider,
    error_rate,
    risk_level,
    output_area2
]))

### PM Insight: Trust is Earned Slowly, Lost Quickly

**Klarna's Journey:**
- Weeks 1-4: AI suggests, humans send (Level 0)
- Weeks 5-8: AI drafts, humans approve (Level 1)
- Weeks 9-16: AI sends routine, humans review sample (Level 2)
- Month 4+: AI handles routine autonomously (Level 3)

**Key metrics to track:**
- Accuracy rate (> 98% for Level 2, > 99% for Level 3)
- Edit rate (how often humans modify AI output)
- Rejection rate (how often humans reject AI actions)
- User satisfaction scores

---
## Demo 3: Human Checkpoint Optimizer

Every checkpoint adds friction but reduces risk. Where's the right balance?

**Configure approval thresholds and see the tradeoffs.**

In [ ]:
#@title Checkpoint Configuration Optimizer

dollar_threshold = widgets.IntSlider(
    value=500,
    min=0,
    max=5000,
    step=100,
    description='$ Threshold:',
    style={'description_width': '100px'},
    continuous_update=False
)

confidence_threshold = widgets.IntSlider(
    value=80,
    min=50,
    max=99,
    step=5,
    description='Confidence %:',
    style={'description_width': '100px'},
    continuous_update=False
)

require_approval = widgets.SelectMultiple(
    options=['Purchases', 'External Emails', 'Data Deletion', 'Refunds', 'Escalations'],
    value=['Purchases', 'Data Deletion', 'Refunds'],
    description='Always Approve:',
    style={'description_width': '100px'}
)

output_area3 = widgets.Output()

def analyze_checkpoints(threshold, confidence, approvals):
    with output_area3:
        clear_output(wait=True)

        # Simulate 100 tasks
        np.random.seed(42)
        n_tasks = 100

        actions = ['Purchases', 'External Emails', 'Data Deletion', 'Refunds',
                   'Escalations', 'Internal Updates', 'Report Generation']
        action_values = [800, 0, 0, 200, 0, 0, 0]  # Dollar values

        results = {'Auto-Approved': 0, 'Human Approved': 0, 'Blocked': 0}
        friction_score = 0

        for _ in range(n_tasks):
            action_idx = np.random.randint(0, len(actions))
            action = actions[action_idx]
            value = action_values[action_idx] * np.random.uniform(0.1, 2)
            conf = np.random.randint(60, 100)

            if action in approvals:
                results['Human Approved'] += 1
                friction_score += 3
            elif value > threshold:
                results['Human Approved'] += 1
                friction_score += 2
            elif conf < confidence:
                results['Human Approved'] += 1
                friction_score += 1
            else:
                results['Auto-Approved'] += 1

        # Calculate metrics
        auto_rate = results['Auto-Approved'] / n_tasks * 100
        human_rate = results['Human Approved'] / n_tasks * 100
        avg_friction = friction_score / n_tasks

        # Risk score (inverse of controls)
        risk_score = auto_rate * 0.5 + (100 - len(approvals) * 20)

        # Create comparison chart
        fig = go.Figure()

        fig.add_trace(go.Bar(
            x=['Auto-Approved', 'Human Approved'],
            y=[results['Auto-Approved'], results['Human Approved']],
            marker_color=['#22c55e', '#f59e0b'],
            text=[f"{results['Auto-Approved']}%", f"{results['Human Approved']}%"],
            textposition='auto'
        ))

        fig.update_layout(
            title="Task Distribution (100 Simulated Tasks)",
            yaxis_title="Number of Tasks",
            height=300
        )
        fig.show()

        # Tradeoff visualization
        fig2 = go.Figure()

        fig2.add_trace(go.Scatterpolar(
            r=[auto_rate, 100-human_rate, 100-avg_friction*10, 100-risk_score*0.5],
            theta=['Speed', 'Efficiency', 'User Experience', 'Safety'],
            fill='toself',
            name='Your Configuration',
            line_color='#40B8A6'
        ))

        fig2.update_layout(
            polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
            title="Configuration Tradeoffs",
            height=400
        )
        fig2.show()

        # Summary
        display(HTML(f"""
        <div style="padding: 15px; background: #f9fafb; border: 1px solid #e5e7eb; margin-top: 10px;">
            <h4 style="margin-top: 0;">Configuration Summary</h4>
            <table style="width: 100%;">
                <tr><td><strong>Auto-Approval Rate:</strong></td><td>{auto_rate:.0f}%</td></tr>
                <tr><td><strong>Human Review Rate:</strong></td><td>{human_rate:.0f}%</td></tr>
                <tr><td><strong>Avg Friction Score:</strong></td><td>{avg_friction:.1f}/3</td></tr>
                <tr><td><strong>Risk Exposure:</strong></td><td>{'Low' if risk_score < 40 else 'Medium' if risk_score < 70 else 'High'}</td></tr>
            </table>
        </div>
        """))

interactive_output3 = widgets.interactive_output(
    analyze_checkpoints,
    {'threshold': dollar_threshold, 'confidence': confidence_threshold, 'approvals': require_approval}
)

display(widgets.VBox([
    widgets.HTML("<h4>Checkpoint Configuration</h4>"),
    dollar_threshold,
    widgets.HTML("<small>Auto-approve purchases below this amount</small>"),
    confidence_threshold,
    widgets.HTML("<small>Require human review if AI confidence below this %</small>"),
    require_approval,
    widgets.HTML("<small>Hold Ctrl/Cmd to select multiple</small>"),
    output_area3
]))

### PM Insight: The Friction vs Safety Tradeoff

| More Checkpoints | Fewer Checkpoints |
|-----------------|-------------------|
| Safer | Faster |
| Slower | Riskier |
| More expensive | Cheaper |
| Better for: high-risk, regulated, new systems | Better for: low-risk, proven systems |

**Common thresholds:**
- Dollar threshold: $100, $500, $1000
- Confidence threshold: 80% (typical), 95%+ (regulated industries)

**Best Practice:** Start with more checkpoints, reduce as the system proves itself.

---
## Demo 4: Agent Failure Mode Detector

What happens when agents go wrong? Learn to recognize the warning signs.

**Explore different failure scenarios.**

In [ ]:
#@title Agent Failure Mode Explorer

failure_modes = {
    'Infinite Loop': {
        'description': 'Agent keeps searching without finding a satisfactory answer',
        'warning_signs': ['Repeated similar tool calls', 'No progress toward goal', 'Iteration count climbing'],
        'prevention': 'Set max iterations (e.g., 10 cycles)',
        'real_example': 'Search agents refining queries forever',
        'risk_level': 60
    },
    'Tool Misuse': {
        'description': 'Agent picks the wrong tool for the task',
        'warning_signs': ['Unexpected results', 'User complaints about wrong actions', 'Tool errors'],
        'prevention': 'Better tool descriptions, validation before execution',
        'real_example': 'Using "delete" instead of "archive"',
        'risk_level': 80
    },
    'Hallucinated Tool': {
        'description': 'Agent tries to use a tool that doesn\'t exist',
        'warning_signs': ['Tool not found errors', 'Agent inventing API calls'],
        'prevention': 'Validate tool exists before calling',
        'real_example': 'Calling translate_api() when it doesn\'t exist',
        'risk_level': 50
    },
    'Data Leak': {
        'description': 'Agent includes sensitive information in external communication',
        'warning_signs': ['PII in outputs', 'Internal data in customer responses'],
        'prevention': 'Output filtering, PII detection, approval for external comms',
        'real_example': 'Air Canada chatbot revealing internal pricing ($800K lawsuit)',
        'risk_level': 95
    },
    'Cascading Errors': {
        'description': 'One error leads to increasingly wrong decisions',
        'warning_signs': ['Compounding mistakes', 'Deviation from expected state'],
        'prevention': 'Step validation, rollback capability, running state checks',
        'real_example': 'Knight Capital: $440M loss in 45 minutes',
        'risk_level': 100
    }
}

failure_dropdown = widgets.Dropdown(
    options=list(failure_modes.keys()),
    value='Infinite Loop',
    description='Failure Mode:',
    style={'description_width': '100px'}
)

output_area4 = widgets.Output()

def show_failure_mode(mode):
    with output_area4:
        clear_output(wait=True)

        info = failure_modes[mode]

        # Risk gauge
        risk = info['risk_level']
        color = '#22c55e' if risk < 50 else '#f59e0b' if risk < 80 else '#dc2626'

        fig = go.Figure(go.Indicator(
            mode="gauge+number",
            value=risk,
            title={'text': "Risk Severity"},
            gauge={
                'axis': {'range': [0, 100]},
                'bar': {'color': color},
                'steps': [
                    {'range': [0, 50], 'color': '#ecfdf5'},
                    {'range': [50, 80], 'color': '#fffbeb'},
                    {'range': [80, 100], 'color': '#fef2f2'}
                ]
            }
        ))
        fig.update_layout(height=250, margin=dict(t=50, b=0))
        fig.show()

        display(HTML(f"""
        <div style="padding: 20px; background: #f9fafb; border: 1px solid #e5e7eb;">
            <h3 style="color: #1A3D4D; margin-top: 0;">{mode}</h3>
            <p><strong>What happens:</strong> {info['description']}</p>

            <h4>Warning Signs</h4>
            <ul>
                {''.join(f'<li>{sign}</li>' for sign in info['warning_signs'])}
            </ul>

            <h4>Prevention</h4>
            <p style="background: #E6F7F5; padding: 10px; border-left: 3px solid #40B8A6;">
                {info['prevention']}
            </p>

            <h4>Real-World Example</h4>
            <p style="background: #fef2f2; padding: 10px; border-left: 3px solid #dc2626;">
                {info['real_example']}
            </p>
        </div>
        """))

interactive_output4 = widgets.interactive_output(
    show_failure_mode,
    {'mode': failure_dropdown}
)

display(widgets.VBox([
    widgets.HTML("<h4>Select a Failure Mode to Explore</h4>"),
    failure_dropdown,
    output_area4
]))

---
## Stakeholder Framing

### How to Explain Agents to Your VP

> "Agents are AI that can take actions, not just give suggestions. Think of them like interns: start supervised, earn trust through success, and always have a manager approve the big decisions."

> "The difference between Knight Capital's disaster and Klarna's success? Human checkpoints at the right moments."

### Key Points for Executive Communication

1. **Trust is earned:** We start with human oversight and reduce it as the AI proves itself
2. **Checkpoints are insurance:** Every approval point is protecting the company from liability
3. **Speed vs safety is a dial:** We can tune it based on business needs
4. **Failure planning is essential:** We need playbooks for when things go wrong

### Questions Your Engineering Team Should Answer

- What's our max iteration limit?
- What actions always require approval?
- What's our rollback capability?
- How do we detect anomalies?
- What's our incident response plan?

---
## Section 5 Summary

### Key Takeaways

1. **Copilot vs Agent:** Start as copilot (suggests), graduate to agent (acts) based on track record

2. **Trust Progression:** Level 0 → Level 3 takes months, not days. Demotion can happen instantly.

3. **Human-in-the-Loop:** Every checkpoint is a tradeoff between safety and speed. Design for the worst case.

4. **Failure Modes:** Know the five failure types - infinite loops, tool misuse, hallucinated tools, data leaks, cascading errors.

### Before You Launch an Agent

- [ ] Max iterations set
- [ ] Tool validation in place
- [ ] Output filtering active
- [ ] Human checkpoints for high-risk actions
- [ ] Rollback capability tested
- [ ] Incident response plan documented

### Next Section: Evaluation

How do you know if your AI is working? How do you measure quality? Section 6 covers evaluation frameworks.